#AGENT MÁY HÚT BỤI BẰNG BFS, DFS

In [117]:
import random
import time
import tkinter as tk
from tkinter import scrolledtext
from collections import deque
import heapq
import math

from IPython.utils import path
from prompt_toolkit.layout.utils import explode_text_fragments

In [118]:
class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action

def get_actions(state):
    """Lấy các hành động hợp lệ từ trạng thái hiện tại"""
    x, y, grid = state
    row, col = len(grid), len(grid[0])
    actions = []

    # NẾU CÓ RÁC -> BẮT BUỘC CHỈ CÓ 1 HÀNH ĐỘNG LÀ DỌN DẸP
    if grid[x][y] == 1:
        return ["CLEAN"]

    # NẾU KHÔNG CÓ RÁC -> MỚI ĐƯỢC PHÉP DI CHUYỂN
    if x > 0 and grid[x-1][y] != 2: actions.append("UP")
    if x < row-1 and grid[x+1][y] != 2: actions.append("DOWN")
    if y > 0 and grid[x][y-1] != 2: actions.append("LEFT")
    if y < col-1 and grid[x][y+1] != 2: actions.append("RIGHT")

    return actions

def get_child_state(state, action):
    """Tính toán trạng thái mới sau khi thực hiện hành động"""
    x, y, grid = state
    if action == "CLEAN":
        new_grid = list(list(r) for r in grid)
        new_grid[x][y] = 0
        return (x, y, tuple(tuple(r) for r in new_grid))
    elif action == "UP":    return (x-1, y, grid)
    elif action == "DOWN":  return (x+1, y, grid)
    elif action == "LEFT":  return (x, y-1, grid)
    elif action == "RIGHT": return (x, y+1, grid)

def goal_test(state):
    """Kiểm tra xem sàn nhà đã sạch hết chưa (không còn số 1)"""
    _, _, grid = state
    for row in grid:
        if 1 in row:
            return False
    return True

def get_solution(node):
    """Truy xuất đường đi từ gốc đến đích"""
    path = []
    while node.parent is not None:
        path.append(node.action)
        node = node.parent
    return path[::-1]

In [119]:
def BFS_Kieu_1(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    reached = set()

    frontier_states = {initial_state}

    while frontier:
        node = frontier.popleft()
        frontier_states.remove(node.state)
        reached.add(node.state)

        if goal_test(node.state):
            return get_solution(node)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in reached and child_state not in frontier_states:
                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [120]:
def BFS_Kieu_2(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    explored = set()
    frontier_states = {initial_state}

    while frontier:
        node = frontier.popleft()
        frontier_states.remove(node.state)
        explored.add(node.state)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in explored and child_state not in frontier_states:
                if goal_test(child_state):
                    return get_solution(child_node)

                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [121]:
def DFS_Kieu_1(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    reached = set()

    frontier_states = {initial_state}

    while frontier:
        node = frontier.pop()
        frontier_states.remove(node.state)
        reached.add(node.state)

        if goal_test(node.state):
            return get_solution(node)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in reached and child_state not in frontier_states:
                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [122]:
def DFS_Kieu_2(initial_state):
    node = Node(initial_state)

    if goal_test(node.state):
        return get_solution(node)

    frontier = deque([node])
    explored = set()
    frontier_states = {initial_state}

    while frontier:
        node = frontier.pop()
        frontier_states.remove(node.state)
        explored.add(node.state)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)
            child_node = Node(child_state, node, action)

            if child_state not in explored and child_state not in frontier_states:
                if goal_test(child_state):
                    return get_solution(child_node)

                frontier.append(child_node)
                frontier_states.add(child_state)

    return "Failure: Không tìm thấy đường đi"

In [123]:
# CÁC HÀM CHO IDS
def get_depth(node):
    """Tính độ sâu của node hiện tại bằng cách đếm ngược về gốc"""
    depth = 0
    current = node
    while current.parent is not None:
        depth += 1
        current = current.parent
    return depth

def is_cycle(node):
    """Kiểm tra xem node hiện tại có tạo thành chu trình không"""
    current = node.parent
    while current is not None:
        if current.state == node.state:
            return True
        current = current.parent
    return False

In [124]:
def DEPTH_LIMITED_SEARCH_1(initial_state, limit):
    node = Node(initial_state)

    frontier = deque([node])
    result = "Failure"

    while frontier:
        node = frontier.pop()

        # Kiểm tra đích
        if goal_test(node.state):
            return get_solution(node)

        # Kiểm tra giới hạn độ sâu
        if get_depth(node) >= limit:
            result = "Cutoff"

        # Kiểm tra chu trình và mở rộng
        elif not is_cycle(node):
            for action in get_actions(node.state):
                child_state = get_child_state(node.state, action)
                child_node = Node(child_state, node, action)
                frontier.append(child_node)

    return result

def ITERATIVE_DEEPENING_SEARCH_1(initial_state, max_depth=100):
    """Thuật toán tìm kiếm sâu dần (IDS)"""
    for depth in range(max_depth):
        result = DEPTH_LIMITED_SEARCH_1(initial_state, depth)

        if result != "Cutoff":
            return result

    return "Failure: Đạt giới hạn độ sâu tối đa mà không tìm thấy đường đi"

In [125]:
def DEPTH_LIMITED_SEARCH_2(initial_state, limit):
    node = Node(initial_state)

    frontier = deque([node])
    result = "Failure"

    while frontier:
        node = frontier.pop()

        # Kiểm tra giới hạn độ sâu
        if get_depth(node) >= limit:
            result = "Cutoff"

        # Kiểm tra chu trình và mở rộng
        elif not is_cycle(node):
            for action in get_actions(node.state):
                child_state = get_child_state(node.state, action)
                child_node = Node(child_state, node, action)
                # Kiểm tra đích
                if goal_test(node.state):
                    return get_solution(node)
                frontier.append(child_node)

    return result

def ITERATIVE_DEEPENING_SEARCH_2(initial_state, max_depth=100):
    """Thuật toán tìm kiếm sâu dần (IDS)"""
    for depth in range(max_depth):
        result = DEPTH_LIMITED_SEARCH_1(initial_state, depth)

        if result != "Cutoff":
            return result

    return "Failure: Đạt giới hạn độ sâu tối đa mà không tìm thấy đường đi"

In [126]:
def count_dirty(state):
    _, _, grid = state

    dirty_count = 0
    for row in grid:
        for cell in row:
            if cell == 1:
                dirty_count += 1
    return dirty_count

In [127]:
def UNIFORM_COST_SEARCH(initial_state):
    node = Node(initial_state)

    frontier = [(0, id(node), node)]
    heapq.heapify(frontier)

    explored = {initial_state: 0}

    while frontier:
        current_cost, _, node = heapq.heappop(frontier)

        if current_cost > explored.get(node.state, float('inf')):
            continue

        if goal_test(node.state):
            return get_solution(node)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)

            step_cost = count_dirty(child_state)
            child_cost = current_cost + step_cost

            if child_state not in explored or child_cost < explored[child_state]:
                explored[child_state] = child_cost
                child_node = Node(child_state, node, action)

                heapq.heappush(frontier, (child_cost, id(child_node), child_node))

    return "Failure: Không tìm thấy đường đi"

In [128]:
def get_heuristic(state):
    x, y, grid = state
    dirts = []

    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == 1:
                dirts.append((r, c))
    if not dirts:
        return 0

    return min(abs(x - dr) + abs(y - dc) for dr, dc in dirts)

In [129]:
def Greedy_Search(initial_state):
    node = Node(initial_state)
    start_h = get_heuristic(initial_state)

    frontier = [(start_h, id(node), node)]
    heapq.heapify(frontier)

    explored = set()

    while frontier:
        current_h, _, node = heapq.heappop(frontier)

        if goal_test(node.state):
            return get_solution(node)

        if node.state in explored:
            continue
        explored.add(node.state)

        for action in get_actions(node.state):
            child_state = get_child_state(node.state, action)

            if child_state not in explored:
                child_h = get_heuristic(child_state)
                child_node = Node(child_state, node, action)

                heapq.heappush(frontier, (child_h, id(child_node), child_node))

    return "Failure: Không tìm thấy đường đi"

In [130]:
def A_STAR_SEARCH(initial_state):
    node = Node(initial_state)

    # Tính toán g(Start) và h(Start)
    g_start = 0
    h_start = get_heuristic(initial_state)
    f_start = g_start + h_start

    frontier = [(f_start, id(node), node)]
    heapq.heapify(frontier)

    explored = {initial_state: g_start}

    while frontier:
        current_f, _, n_node = heapq.heappop(frontier)
        n_state = n_node.state

        current_g = current_f - get_heuristic(n_state)

        if current_g > explored.get(n_state, float('inf')):
            continue

        if goal_test(n_state):
            return get_solution(n_node)

        for action in get_actions(n_state):
            m_state = get_child_state(n_state, action)

            # Tính g_new(m) = g(n) + cost(n, m)
            g_new_m = current_g + count_dirty(m_state)

            # Tính h(m)
            h_m = get_heuristic(m_state)

            # Tính f(m)
            f_new_m = g_new_m + h_m

            # Nếu m chưa từng xuất hiện HOẶC tìm thấy đường đi mới có chi phí g tốt hơn
            if m_state not in explored or g_new_m < explored[m_state]:
                # Cập nhật chi phí g tốt nhất cho trạng thái m
                explored[m_state] = g_new_m

                m_node = Node(m_state, n_node, action)

                heapq.heappush(frontier, (f_new_m, id(m_node), m_node))

    return "Failure: Không tìm thấy đường đi"

In [131]:
def COST_LIMITED_SEARCH(initial_state, cost):

    node = Node(initial_state)

    # Cấu trúc (node, g_score)
    g_start = 0
    frontier = deque([(node, g_start)])

    next_cost = float('inf')

    cutoff_occurred = False

    while frontier:
        node, current_g = frontier.pop()
        state = node.state

        # Tính f(n) = g(n) + h(n)
        f_score = current_g + get_heuristic(state)

        if f_score > cost:
            cutoff_occurred = True

            if f_score < next_cost:
                next_cost = f_score
            continue

        if goal_test(state):
            return get_solution(node), cost

        if not is_cycle(node):

            for action in reversed(get_actions(state)):
                child_state = get_child_state(state, action)
                child_node = Node(child_state, node, action)

                g_new = current_g + count_dirty(child_state)

                frontier.append((child_node, g_new))

    if cutoff_occurred:
        return "Cutoff", next_cost

    return "Failure", float('inf')

def ITERATIVE_DEEPENING_A_STAR(initial_state):
    cost = get_heuristic(initial_state)

    while True:
        result, next_cost = COST_LIMITED_SEARCH(initial_state, cost)

        if result != "Cutoff" and result != "Failure":
            return result

        if result == "Failure":
            return "Failure: Không tìm thấy đường đi"

        if next_cost == float('inf'):
            return "Failure: Không tìm thấy đường đi"

        cost = next_cost

In [132]:
def get_cost(state):
    x, y, grid = state
    dirty_cnt = count_dirty(state)

    dirts = []
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == 1:
                dirts.append((r, c))

    if not dirts:
        return 0

    min_distance = min(abs(x - dr) + abs(y - dc) for dr, dc in dirts)

    return (dirty_cnt * 100) + min_distance

In [133]:
def SIMPLE_HILL_CLIMBING(initial_state):
    current_state = initial_state
    current_node = Node(current_state)

    while True:
        if goal_test(current_state):
            return get_solution(current_node)

        neighbor_found = False

        for action in get_actions(current_state):
            next_state = get_child_state(current_state, action)

            if get_cost(next_state) < get_cost(current_state):
                next_node = Node(next_state, current_node, action)

                current_state = next_state
                current_node = next_node

                neighbor_found = True
                break

        if not neighbor_found:
            print("Dừng vì đạt cực tiểu cục bộ!")
            return get_solution(current_node)

In [134]:
def STEEPEST_ASCENT_HILL_CLIMBING(initial_state):
    current_state = initial_state
    current_node = Node(current_state)

    while True:
        if goal_test(current_state):
            return get_solution(current_node)

        actions = get_actions(current_state)

        best_state = None
        best_node = None
        min_neighbor_cost = float('inf')

        for action in actions:
            next_state = get_child_state(current_state, action)
            next_cost = get_cost(next_state)

            if next_cost < min_neighbor_cost:
                min_neighbor_cost = next_cost
                best_state = next_state

                best_node = Node(next_state, current_node, action)

        if best_state is not None and get_cost(best_state) < get_cost(current_state):
            current_state = best_state
            current_node = best_node
        else:
            print("Dừng vì đạt cực đại cục bộ!")
            return get_solution(current_node)

In [135]:
def STOCHASTIC_HILL_CLIMBING(initial_state):
    current_state = initial_state
    current_node = Node(current_state)

    while True:
        if goal_test(current_state):
            return get_solution(current_node)

        better_neighbors = []

        for action in get_actions(current_state):
            next_state = get_child_state(current_state, action)

            if get_cost(next_state) < get_cost(current_state):
                next_node = Node(next_state, current_node, action)

                better_neighbors.append((next_state, next_node))

        if not better_neighbors:
            print("Dừng vì đạt cực tiểu/cực đại cục bộ!")
            return get_solution(current_node)

        else:
            min_cost = min(get_cost(state) for state, node in better_neighbors)

            best_candidates = [(state, node) for state, node in better_neighbors if get_cost(state) == min_cost]

            chosen_state, chosen_node = random.choice(best_candidates)

            current_state = chosen_state
            current_node = chosen_node

In [136]:
def RANDOM_RESTART_HILL_CLIMBING(initial_state, max_restart=10):

    for i in range(1, max_restart + 1):
        current_state = initial_state
        current_node = Node(current_state)

        print(f"Lượt {i}: Khởi động lại từ vị trí ban đầu ({current_state[0]}, {current_state[1]})")

        while True:
            if goal_test(current_state):
                return get_solution(current_node)

            better_neighbors = []

            for action in get_actions(current_state):
                next_state = get_child_state(current_state, action)

                if get_cost(next_state) < get_cost(current_state):

                    next_node = Node(next_state, current_node, action)
                    better_neighbors.append((next_state, next_node))

            if not better_neighbors:
                print(f"Lượt {i} bị kẹt cực tiểu cục bộ. Đổi lượt restart!")
                break

            else:
                min_cost = min(get_cost(state) for state, node in better_neighbors)

                best_candidates = [(state, node) for state, node in better_neighbors if get_cost(state) == min_cost]

                chosen_state, chosen_node = random.choice(best_candidates)

                current_state = chosen_state
                current_node = chosen_node

    return "Failure: Chạy hết sạch MAX_RESTART lượt mà không chạm được Goal"

In [137]:
def LOCAL_BEAM_SEARCH(initial_state, k=3):
    current_node_set = [Node(initial_state)]

    print(f"Khởi tạo Local Beam Search với chùm k = {k} từ vị trí ban đầu ---")

    while True:
        neighbor_nodes = []

        for node in current_node_set:
            state = node.state

            for action in get_actions(state):
                child_state = get_child_state(state, action)
                child_node = Node(child_state, node, action)

                neighbor_nodes.append(child_node)

        if not neighbor_nodes:
            print("Dừng vì tất cả các nhánh trong chùm không còn đường đi!")
            return []

        for node in neighbor_nodes:
            if goal_test(node.state):
                return get_solution(node)

        neighbor_nodes.sort(key=lambda n: get_cost(n.state))

        current_node_set = neighbor_nodes[:k]

In [138]:
def SIMULATED_ANNEALING(initial_state, T0=100.0, Tmin=0.1, alpha=0.95):
    # current state = start
    current_state = initial_state
    current_node = Node(current_state)

    T = T0

    while T > Tmin:
        if goal_test(current_state):
            return get_solution(current_node)

        actions = get_actions(current_state)
        if not actions:
            break

        chosen_action = random.choice(actions)
        next_state = get_child_state(current_state, chosen_action)
        next_node = Node(next_state, current_node, chosen_action)

        delta = get_cost(next_state) - get_cost(current_state)

        if delta < 0:
            current_state = next_state
            current_node = next_node
        else:
            p = math.exp(-delta / T)

            if random.random() < p:
                current_state = next_state
                current_node = next_node

        T = alpha * T

    return get_solution(current_node)

In [139]:
class BeliefNode:
    def __init__(self, belief_state, parent=None, action=None):
        self.belief_state = belief_state # Là một tuple lưu các trạng thái khả thi
        self.parent = parent
        self.action = action

def get_child_belief_state(belief_state, action):
    next_physical_states = set()

    for p_state in belief_state:
        x, y, grid = p_state
        row, col = len(grid), len(grid[0])

        if action == "CLEAN":
            if grid[x][y] == 1:
                new_grid = list(list(r) for r in grid)
                new_grid[x][y] = 0
                next_physical_states.add((x, y, tuple(tuple(r) for r in new_grid)))
            else:
                next_physical_states.add(p_state)
        elif action == "UP":
            new_x = x - 1 if (x > 0 and grid[x-1][y] != 2) else x
            next_physical_states.add((new_x, y, grid))
        elif action == "DOWN":
            new_x = x + 1 if (x < row - 1 and grid[x+1][y] != 2) else x
            next_physical_states.add((new_x, y, grid))
        elif action == "LEFT":
            new_y = y - 1 if (y > 0 and grid[x][y-1] != 2) else y
            next_physical_states.add((x, new_y, grid))
        elif action == "RIGHT":
            new_y = y + 1 if (y < col - 1 and grid[x][y+1] != 2) else y
            next_physical_states.add((x, new_y, grid))

    return tuple(sorted(list(next_physical_states)))

def get_belief_solution(node):
    path = []
    current = node
    while current.parent is not None:
        path.append(current.action)
        current = current.parent
    return path[::-1]

def belief_goal_test(belief_state):
    for p_state in belief_state:
        _, _, grid = p_state
        for row in grid:
            if 1 in row:
                return False
    return True

In [140]:
def BELIEF_SEARCH_BFS(initial_physical_state):
    _, _, initial_grid = initial_physical_state
    row, col = len(initial_grid), len(initial_grid[0])

    initial_belief_set = set()
    for r in range(row):
        for c in range(col):
            if initial_grid[r][c] != 2:
                initial_belief_set.add((r, c, initial_grid))
    initial_belief = tuple(sorted(list(initial_belief_set)))

    root_node = BeliefNode(initial_belief)

    if belief_goal_test(root_node.belief_state):
        return get_belief_solution(root_node)

    frontier = deque([root_node])
    reached = {initial_belief}

    while frontier:
        node = frontier.popleft()

        has_dirt_somewhere = False
        all_have_dirt = True

        for p_state in node.belief_state:
            x, y, grid = p_state
            if grid[x][y] == 1:
                has_dirt_somewhere = True
            else:
                all_have_dirt = False

        if all_have_dirt:
            possible_actions = ["CLEAN"]
        elif has_dirt_somewhere:
            possible_actions = ["CLEAN", "UP", "DOWN", "LEFT", "RIGHT"]
        else:
            possible_actions = ["UP", "DOWN", "LEFT", "RIGHT"]

        for action in possible_actions:
            child_belief = get_child_belief_state(node.belief_state, action)

            if child_belief not in reached:
                child_node = BeliefNode(child_belief, node, action)

                if belief_goal_test(child_belief):
                    return get_belief_solution(child_node)

                reached.add(child_belief)
                frontier.append(child_node)

    return "Failure: Không tìm thấy đường đi"

In [141]:
def bfs_find_target(start_x, start_y, agent_grid):
    row, col = len(agent_grid), len(agent_grid[0])

    # GIAI ĐOẠN 1: Quét tìm ô có RÁC (1) gần nhất đã lộ diện
    queue = deque([(start_x, start_y, [])])
    visited = {(start_x, start_y)}
    while queue:
        x, y, path = queue.popleft()
        if agent_grid[x][y] == 1:
            return path
        for action, dx, dy in [("UP", -1, 0), ("DOWN", 1, 0), ("LEFT", 0, -1), ("RIGHT", 0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < row and 0 <= ny < col:
                if agent_grid[nx][ny] != 2 and (nx, ny) not in visited:
                    visited.add((nx, ny))
                    queue.append((nx, ny, path + [action]))

    # GIAI ĐOẠN 2: Nếu không thấy ô rác nào, tìm ô SƯƠNG MÙ (-1) gần nhất
    queue = deque([(start_x, start_y, [])])
    visited = {(start_x, start_y)}
    while queue:
        x, y, path = queue.popleft()
        if agent_grid[x][y] == -1:
            return path
        for action, dx, dy in [("UP", -1, 0), ("DOWN", 1, 0), ("LEFT", 0, -1), ("RIGHT", 0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < row and 0 <= ny < col:
                if agent_grid[nx][ny] != 2 and (nx, ny) not in visited:
                    visited.add((nx, ny))
                    queue.append((nx, ny, path + [action]))
    return []

In [142]:
def PARTIAL_BELIEF_SEARCH(initial_physical_state):
    start_x, start_y, initial_grid = initial_physical_state
    row, col = len(initial_grid), len(initial_grid[0])

    true_grid = [list(r) for r in initial_grid]

    agent_grid = []
    for r in range(row):
        temp_row = []
        for c in range(col):
            if true_grid[r][c] == 2:
                temp_row.append(2)
            else:
                temp_row.append(-1)
        agent_grid.append(temp_row)

    path_taken = []
    x, y = start_x, start_y
    current_plan = []

    for step in range(150):
        agent_grid[x][y] = true_grid[x][y]

        is_done = True
        for r in range(row):
            for c in range(col):
                if agent_grid[r][c] == 1 or agent_grid[r][c] == -1:
                    is_done = False
                    break
            if not is_done: break

        if is_done:
            return path_taken

        if agent_grid[x][y] == 1:
            action = "CLEAN"
            agent_grid[x][y] = 0
            true_grid[x][y] = 0
            current_plan = []
        else:
            if not current_plan:
                current_plan = bfs_find_target(x, y, agent_grid)

            if not current_plan:
                return "Failure: Robot bị kẹt, không thể khám phá phần còn lại."

            action = current_plan.pop(0) 

        path_taken.append(action)
        if action == "UP":      x -= 1
        elif action == "DOWN":  x += 1
        elif action == "LEFT":  y -= 1
        elif action == "RIGHT": y += 1

    return "Failure: Vượt quá giới hạn vòng lặp mô phỏng"

In [143]:
def extract_linear_path(plan):
    if not plan or plan == "failure":
        return []

    path = []
    current = plan

    while current and isinstance(current, list) and len(current) == 2:
        action, next_part = current
        path.append(action)

        if isinstance(next_part, dict):
            if next_part:
                current = list(next_part.values())[0]
            else:
                break
        else:
            current = next_part

    return path

In [144]:
def get_nondeterministic_results(state, action):
    primary_child = get_child_state(state, action)
    results = [primary_child]

    return tuple(set(results))

def AND_OR_GRAPH_SEARCH(initial_state):
    return OR_SEARCH(initial_state, [])

def OR_SEARCH(state, path):
    if goal_test(state):
        return []
    if state in path:
        return "failure"

    actions = get_actions(state)
    for action in actions:
        results = get_nondeterministic_results(state, action)
        plan = AND_SEARCH(results, path + [state])
        if plan != "failure":
            return [action, plan]
    return "failure"

def AND_SEARCH(states, path):
    plans = {}
    for s in states:
        plan_s = OR_SEARCH(s, path)
        if plan_s == "failure":
            return "failure"
        plans[s] = plan_s
    return plans

In [145]:
class VacuumApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Agent Máy Hút Bụi - Hệ Thống Mô Phỏng Thuật Toán")
        self.root.state('zoomed')
        self.root.configure(bg="#F5F6F8") # Nền sáng hiện đại

        self.row, self.col = 3, 3
        self.cell_size = 120

        self.is_animating = False
        self.is_paused = False

        self.setup_ui()
        self.generate_new_environment()

    def setup_ui(self):
        # ----------------- PANEL ĐIỀU KHIỂN (BÊN TRÁI) -----------------
        left_frame = tk.Frame(self.root, width=280, bg="#FFFFFF", relief=tk.FLAT, bd=0)
        left_frame.pack(side=tk.LEFT, fill=tk.Y, padx=10, pady=10)

        # Tiêu đề chính
        tk.Label(left_frame, text="BẢNG ĐIỀU KHIỂN", font=("Segoe UI", 13, "bold"), bg="#FFFFFF", fg="#2C3E50").pack(pady=(15, 10))

        # Khởi tạo biến lưu thuật toán lựa chọn
        self.algo_var = tk.StringVar(value="BFS1")

        # --- Nhóm 1: Tìm kiếm mù (Uninformed) ---
        group_blind = tk.LabelFrame(left_frame, text=" Tìm Kiếm Không Có Thông Tin ", font=("Segoe UI", 9, "bold"), bg="#FFFFFF", fg="#2980B9")
        group_blind.pack(fill=tk.X, padx=10, pady=5)

        blind_algos = [("BFS Kiểu 1", "BFS1"), ("BFS Kiểu 2", "BFS2"),
                       ("DFS Kiểu 1", "DFS1"), ("DFS Kiểu 2", "DFS2"),
                       ("IDS Kiểu 1", "IDS1"), ("IDS Kiểu 2", "IDS2"),
                       ("Uniform Cost (UCS)", "UCS")]
        for text, val in blind_algos:
            tk.Radiobutton(group_blind, text=text, variable=self.algo_var, value=val, bg="#FFFFFF", font=("Segoe UI", 9), activebackground="#FFFFFF").pack(anchor=tk.W, padx=10, pady=1)

        # --- Nhóm 2: Tìm kiếm Heuristic (Informed) ---
        group_heuristic = tk.LabelFrame(left_frame, text=" Tìm Kiếm Có Thông Tin ", font=("Segoe UI", 9, "bold"), bg="#FFFFFF", fg="#2980B9")
        group_heuristic.pack(fill=tk.X, padx=10, pady=5)

        heuristic_algos = [("Greedy Search", "Greedy"), ("A* Search", "AStar"), ("IDA* Search", "IDA*")]
        for text, val in heuristic_algos:
            tk.Radiobutton(group_heuristic, text=text, variable=self.algo_var, value=val, bg="#FFFFFF", font=("Segoe UI", 9), activebackground="#FFFFFF").pack(anchor=tk.W, padx=10, pady=1)

        # --- Nhóm 3: Chiến lược Leo Đồi ---
        group_hill = tk.LabelFrame(left_frame, text=" Giải Thuật Tìm Kiếm Địa Phương ", font=("Segoe UI", 9, "bold"), bg="#FFFFFF", fg="#2980B9")
        group_hill.pack(fill=tk.X, padx=10, pady=5)

        local_algos = [("Simple Hill Climbing", "simple"), ("Steepest Ascent HC", "steepest"), ("Stochastic HC", "stochastic"), ("Random Restart HC", "random_restart"), ("Local Beam Search", "beam"), ("Simulated Annealing", "annealing")]
        for text, val in local_algos:
            tk.Radiobutton(group_hill, text=text, variable=self.algo_var, value=val, bg="#FFFFFF", font=("Segoe UI", 9), activebackground="#FFFFFF").pack(anchor=tk.W, padx=10, pady=1)

        # --- NHÓM 4: TÌM KIẾM KHÔNG GIAN NIỀM TIN (BELIEF SPACE) ---
        group_belief = tk.LabelFrame(left_frame, text=" Tìm Kiếm Không Gian Niềm Tin ", font=("Segoe UI", 9, "bold"), bg="#FFFFFF", fg="#2980B9")
        group_belief.pack(fill=tk.X, padx=10, pady=5)

        belief_algos = [("Toàn Phần", "BeliefBFS"),
                        ("Một Phần", "Partial"),]
        for text, val in belief_algos:
            tk.Radiobutton(group_belief, text=text, variable=self.algo_var, value=val, bg="#FFFFFF", font=("Segoe UI", 9), activebackground="#FFFFFF").pack(anchor=tk.W, padx=10, pady=2)

        # --- NHÓM 5: TÌM KIẾM TRÊN MÔI TRƯỜNG BẤT ĐỊNH ---
        group_and_or = tk.LabelFrame(left_frame, text=" Môi Trường Bất Định ", font=("Segoe UI", 9, "bold"), bg="#FFFFFF", fg="#E67E22")
        group_and_or.pack(fill=tk.X, padx=10, pady=5)

        tk.Radiobutton(
            group_and_or,
            text="AND-OR Graph Search",
            variable=self.algo_var,
            value="AND_OR",
            bg="#FFFFFF",
            font=("Segoe UI", 9),
            activebackground="#FFFFFF"
        ).pack(anchor=tk.W, padx=10, pady=5)

        # Cấu hình nhập Độ sâu IDS
        config_frame = tk.Frame(left_frame, bg="#FFFFFF")
        config_frame.pack(fill=tk.X, padx=10, pady=(10, 5))
        tk.Label(config_frame, text="Giới hạn độ sâu IDS:", bg="#FFFFFF", font=("Segoe UI", 9, "italic"), fg="#555555").pack(side=tk.LEFT, padx=5)
        self.depth_entry = tk.Entry(config_frame, width=8, font=("Segoe UI", 9), justify=tk.CENTER)
        self.depth_entry.insert(0, "20")
        self.depth_entry.pack(side=tk.LEFT, padx=5)

        # --- Hệ thống các nút bấm chức năng điều khiển ---
        self.gen_btn = tk.Button(left_frame, text="🔄 Tạo Sàn Mới", command=self.generate_new_environment, width=24, bg="#F39C12", fg="white", font=("Segoe UI", 10, "bold"), relief=tk.FLAT, activebackground="#E67E22", activeforeground="white")
        self.gen_btn.pack(pady=(15, 5))

        self.start_btn = tk.Button(left_frame, text="▶ Chạy Mô Phỏng", command=self.start_simulation, width=24, bg="#2ECC71", fg="white", font=("Segoe UI", 10, "bold"), relief=tk.FLAT, activebackground="#27AE60", activeforeground="white")
        self.start_btn.pack(pady=4)

        self.pause_btn = tk.Button(left_frame, text="⏸ Tạm Dừng", command=self.toggle_pause, width=24, bg="#3498DB", fg="white", font=("Segoe UI", 10, "bold"), relief=tk.FLAT, state=tk.DISABLED, activebackground="#2980B9", activeforeground="white")
        self.pause_btn.pack(pady=4)

        self.stop_btn = tk.Button(left_frame, text="⏹ Kết Thúc", command=self.stop_simulation, width=24, bg="#E74C3C", fg="white", font=("Segoe UI", 10, "bold"), relief=tk.FLAT, state=tk.DISABLED, activebackground="#C0392B", activeforeground="white")
        self.stop_btn.pack(pady=4)


        # ----------------- PANEL HIỂN THỊ KẾT QUẢ (BÊN DƯỚI) -----------------
        bottom_frame = tk.Frame(self.root, height=110, bg="#FFFFFF", relief=tk.FLAT, bd=0)
        bottom_frame.pack(side=tk.BOTTOM, fill=tk.X, padx=10, pady=(5, 10))

        tk.Label(bottom_frame, text="KẾT QUẢ ĐƯỜNG ĐI CHI TIẾT:", font=("Segoe UI", 10, "bold"), bg="#FFFFFF", fg="#2C3E50").pack(anchor=tk.W, padx=10, pady=(8, 2))
        self.result_text = tk.Text(bottom_frame, height=3, wrap=tk.WORD, font=("Consolas", 10), state=tk.DISABLED, bg="#F8F9FA", relief=tk.SOLID, bd=1)
        self.result_text.pack(fill=tk.BOTH, expand=True, padx=10, pady=(0, 10))


        # ----------------- PANEL LOG HOẠT ĐỘNG (BÊN PHẢI) -----------------
        right_frame = tk.Frame(self.root, width=280, bg="#FFFFFF", relief=tk.FLAT, bd=0)
        right_frame.pack(side=tk.RIGHT, fill=tk.Y, padx=10, pady=10)

        tk.Label(right_frame, text="BẢNG LOG QUÁ TRÌNH CHẠY", font=("Segoe UI", 11, "bold"), bg="#FFFFFF", fg="#2C3E50").pack(pady=(15, 5))
        self.log_area = scrolledtext.ScrolledText(right_frame, width=32, wrap=tk.WORD, font=("Consolas", 9), bg="#F8F9FA", relief=tk.SOLID, bd=1)
        self.log_area.pack(fill=tk.BOTH, expand=True, padx=10, pady=(0, 10))


        # ----------------- KHU VỰC TRUNG TÂM (ĐỒ HỌA CHÍNH MA TRẬN) -----------------
        center_frame = tk.Frame(self.root, bg="#F5F6F8")
        center_frame.pack(side=tk.LEFT, expand=True, fill=tk.BOTH)

        # Khung bọc Canvas tạo cảm giác nổi khối
        canvas_container = tk.Frame(center_frame, bg="#FFFFFF", padx=10, pady=10, relief=tk.SOLID, bd=1)
        canvas_container.pack(expand=True)

        self.canvas = tk.Canvas(canvas_container, width=self.col*self.cell_size, height=self.row*self.cell_size, bg="#FFFFFF", highlightthickness=0)
        self.canvas.pack()

    # --- CÁC HÀM XỬ LÝ LOGIC NÚT BẤM (Giữ nguyên cấu trúc của bạn) ---
    def toggle_pause(self):
        if not self.is_animating: return
        self.is_paused = not self.is_paused

        if self.is_paused:
            self.pause_btn.config(text="▶ Tiếp Tục", bg="#2980B9")
            self.log(">>> ĐÃ TẠM DỪNG MÔ PHỎNG.")
        else:
            self.pause_btn.config(text="⏸ Tạm Dừng", bg="#3498DB")
            self.log(">>> TIẾP TỤC MÔ PHỎNG.")

    def stop_simulation(self):
        if not self.is_animating: return
        self.is_animating = False
        self.is_paused = False

        self.log(">>> ĐÃ HỦY VÀ KẾT THÚC MÔ PHỎNG NGANG CHỪNG.")
        self.reset_button_states()

    def reset_button_states(self):
        self.start_btn.config(state=tk.NORMAL)
        self.gen_btn.config(state=tk.NORMAL)
        self.pause_btn.config(state=tk.DISABLED, text="⏸ Tạm Dừng", bg="#3498DB")
        self.stop_btn.config(state=tk.DISABLED)

    def log(self, message):
        self.log_area.insert(tk.END, message + "\n")
        self.log_area.see(tk.END)

    def update_result(self, message):
        self.result_text.config(state=tk.NORMAL)
        self.result_text.delete(1.0, tk.END)
        self.result_text.insert(tk.END, message)
        self.result_text.config(state=tk.DISABLED)

    def generate_new_environment(self):
        if self.is_animating: return

        #self.mat = [[1, 0, 1, 1], [0, 1, 0, 0], [1, 0, 1, 1], [1, 0, 1, 0]]
        self.mat = [[random.randint(0, 1) for _ in range(self.col)] for _ in range(self.row)]
        while True:
            self.start_x, self.start_y = random.randint(0, self.row-1), random.randint(0, self.col-1)
            if self.mat[self.start_x][self.start_y] != 2:
                break

        self.current_x, self.current_y = self.start_x, self.start_y
        self.current_grid = [list(r) for r in self.mat]

        self.log_area.delete(1.0, tk.END)
        self.log("Đã khởi tạo môi trường thử nghiệm mới.")
        self.update_result("Chưa chạy thuật toán.")
        self.draw_grid()

    def draw_grid(self):
        self.canvas.delete("all")
        for r in range(self.row):
            for c in range(self.col):
                x1, y1 = c * self.cell_size, r * self.cell_size
                x2, y2 = x1 + self.cell_size, y1 + self.cell_size

                val = self.current_grid[r][c]
                # Màu sắc phẳng hiện đại: Sàn trống màu xám nhạt, Ô rác màu nâu đất nhẹ nhàng
                color = "#FFFFFF" if val == 0 else "#EDBB99" if val == 1 else "#2C3E50"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="#BDC3C7", width=1)

                if val == 1:
                    self.canvas.create_text(x1+self.cell_size/2, y1+self.cell_size/2, text="🗑️", font=("Arial", 22))

                if r == self.current_x and c == self.current_y:
                    padding = 8
                    self.canvas.create_oval(x1+padding, y1+padding, x2-padding, y2-padding, fill="#E0F7FA", outline="#00BCD4", width=2)
                    self.canvas.create_text(x1+self.cell_size/2, y1+self.cell_size/2, text="🤖", font=("Arial", 22))

    def start_simulation(self):
        if self.is_animating: return

        self.is_paused = False
        self.current_x, self.current_y = self.start_x, self.start_y
        self.current_grid = [list(r) for r in self.mat]
        self.draw_grid()
        self.log_area.delete(1.0, tk.END)

        initial_state = (self.start_x, self.start_y, tuple(tuple(r) for r in self.mat))
        algo_choice = self.algo_var.get()

        try:
            depth_limit = int(self.depth_entry.get())
        except ValueError:
            depth_limit = 20

        self.log(f"Đang tiến hành tìm đường bằng: {algo_choice}...")
        self.root.update()

        start_time = time.time()

        # Áp dụng bộ điều phối 13 thuật toán đầy đủ
        if algo_choice == "BFS1": path = BFS_Kieu_1(initial_state)
        elif algo_choice == "BFS2": path = BFS_Kieu_2(initial_state)
        elif algo_choice == "DFS1": path = DFS_Kieu_1(initial_state)
        elif algo_choice == "DFS2": path = DFS_Kieu_2(initial_state)
        elif algo_choice == "UCS": path = UNIFORM_COST_SEARCH(initial_state)
        elif algo_choice == "IDS1": path = ITERATIVE_DEEPENING_SEARCH_1(initial_state, max_depth=depth_limit)
        elif algo_choice == "IDS2": path = ITERATIVE_DEEPENING_SEARCH_2(initial_state, max_depth=depth_limit)
        elif algo_choice == "Greedy": path = Greedy_Search(initial_state)
        elif algo_choice == "AStar": path = A_STAR_SEARCH(initial_state)
        elif algo_choice == "IDA*": path = ITERATIVE_DEEPENING_A_STAR(initial_state)
        elif algo_choice == "simple": path = SIMPLE_HILL_CLIMBING(initial_state)
        elif algo_choice == "steepest": path = STEEPEST_ASCENT_HILL_CLIMBING(initial_state)
        elif algo_choice == "stochastic": path = STEEPEST_ASCENT_HILL_CLIMBING(initial_state)
        elif algo_choice == "random_restart": path = RANDOM_RESTART_HILL_CLIMBING(initial_state, max_restart=depth_limit)
        elif algo_choice == "beam": path = LOCAL_BEAM_SEARCH(initial_state, k=2)
        elif algo_choice == "annealing": path = SIMULATED_ANNEALING(initial_state, T0=100.0, Tmin=0.1, alpha=0.95)
        elif algo_choice == "BeliefBFS": path = BELIEF_SEARCH_BFS(initial_state)
        elif algo_choice == "Partial": path = PARTIAL_BELIEF_SEARCH(initial_state)
        elif algo_choice == "AND_OR":
            # 1. Chạy thuật toán tìm kiếm AND-OR ngầm
            raw_tree_plan = AND_OR_GRAPH_SEARCH(initial_state)

            if raw_tree_plan == "failure" or raw_tree_plan is None:
                self.log("AND-OR: Không tìm thấy đường đi an toàn!")
                path = []
            else:
                # 2. Trích xuất chuỗi hành động tuyến tính đưa vào biến path, KHÔNG in cây rườm rà
                path = extract_linear_path(raw_tree_plan)
                self.log(f"AND-OR tìm thấy lộ trình: {path}")
        else: path = []

        elapsed = time.time() - start_time

        if isinstance(path, str) or path is None:
            msg = path if isinstance(path, str) else "Failure"
            self.log(f"Thất bại! {msg}")
            self.update_result(msg)
        else:
            self.log(f"Tìm thấy đường đi thành công! (Thời gian: {elapsed:.4f}s)")
            self.log(f"Tổng số bước đi tối ưu: {len(path)}")
            self.update_result(" -> ".join(path) if path else "Sàn sạch sẽ hoàn toàn ngay từ đầu, Agent không cần di chuyển.")

            if len(path) > 0:
                self.is_animating = True
                self.start_btn.config(state=tk.DISABLED)
                self.gen_btn.config(state=tk.DISABLED)
                self.pause_btn.config(state=tk.NORMAL)
                self.stop_btn.config(state=tk.NORMAL)

                self.animate_step(path, 0)

    def animate_step(self, path, step_idx):
        if not self.is_animating:
            return

        if self.is_paused:
            self.root.after(200, self.animate_step, path, step_idx)
            return

        if step_idx >= len(path):
            self.is_animating = False
            self.reset_button_states()
            self.log(">>> HOÀN THÀNH QUÁ TRÌNH DỌN DẸP TOÀN BỘ SÀN NHÀ!")
            return

        action = path[step_idx]
        self.log(f"Bước {step_idx + 1}: [{action}]")

        # CẬP NHẬT ĐỒ HỌA CÓ CHẶN BIÊN AN TOÀN ĐỂ ROBOT KHÔNG BIẾN MẤT
        if action == "CLEAN":
            self.current_grid[self.current_x][self.current_y] = 0
        elif action == "UP":
            # Chỉ cho phép giảm x nếu chưa chạm biên trên (hàng 0)
            if self.current_x > 0: self.current_x -= 1
        elif action == "DOWN":
            # Chỉ cho phép tăng x nếu chưa chạm biên dưới (hàng row-1)
            if self.current_x < self.row - 1: self.current_x += 1
        elif action == "LEFT":
            # Chỉ cho phép giảm y nếu chưa chạm biên trái (cột 0)
            if self.current_y > 0: self.current_y -= 1
        elif action == "RIGHT":
            # Chỉ cho phép tăng y nếu chưa chạm biên phải (cột col-1)
            if self.current_y < self.col - 1: self.current_y += 1

        self.draw_grid()
        self.root.after(250, self.animate_step, path, step_idx + 1)

if __name__ == "__main__":
    root = tk.Tk()
    app = VacuumApp(root)
    root.mainloop()